# 02 — Graph Construction (CholecT50)

**Goal:** Build causal knowledge graphs from the parsed surgical action triplets.
Each graph models the relationships between instruments and anatomical targets,
connected by surgical verbs (actions).

**Videos:** VID01, VID05, VID40

## Graph Schema Design

### Node Types

We have two categories of nodes, forming a **bipartite-like** graph:

| Node Type | Source | Examples | Attributes |
|-----------|--------|----------|------------|
| **Instrument** | `instrument` column | grasper, hook, bipolar, clipper, scissors, irrigator | `node_type='instrument'`, `id` |
| **Target** | `target` column | gallbladder, cystic_plate, cystic_duct, liver, omentum, ... | `node_type='target'`, `id` |

### Edge Definition

Edges connect an **instrument node** to a **target node**. The edge label is
the **verb** (surgical action). Since the same instrument–target pair can be
connected by different verbs, we use a **MultiDiGraph** (directed, allows
parallel edges with different keys).

```
  [grasper] --retract--> [gallbladder]
  [grasper] --grasp----> [gallbladder]
  [hook]    --dissect--> [cystic_plate]
```

### Edge Attributes

| Attribute | Type | Description |
|-----------|------|-------------|
| `verb` | str | The surgical action (edge label / key) |
| `triplet_id` | int | CholecT50 triplet class ID |
| `triplet_label` | str | Full triplet string |
| `frequency` | int | Number of frames where this triplet appears |
| `frame_list` | list[int] | Ordered list of frame IDs |
| `first_frame` / `last_frame` | int | Temporal boundaries |
| `duration_frames` | int | Span from first to last frame |
| `timestamp_start` / `timestamp_end` | float | Time in seconds |

### Summary

```
  [INSTRUMENT] --verb--> [TARGET]
     attrs: frequency, frame_list, timestamps
  NetworkX: nx.MultiDiGraph
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import networkx as nx
from pathlib import Path
from src.graph_builder import (
    build_graph,
    build_graphs_for_videos,
    graph_stats,
    print_graph_stats,
    save_graph,
)

# Project paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
GRAPH_DIR = PROJECT_ROOT / 'outputs' / 'graphs'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print('Parsed CSVs:', sorted(os.listdir(TRIPLET_DIR)))

Parsed CSVs: ['1_triplets.csv', '40_triplets.csv', '5_triplets.csv', 'VID01_triplets.csv', 'VID05_triplets.csv', 'VID40_triplets.csv', 'all_triplets.csv']


## Build Graph for a Single Video (VID01)

In [2]:
df_vid01 = pd.read_csv(TRIPLET_DIR / 'VID01_triplets.csv')
G_vid01 = build_graph(df_vid01, video_name='VID01')
print_graph_stats(G_vid01)


=== VID01 Graph Stats ===
Nodes: 15 (6 instruments + 9 targets)
Edges: 19
Most frequent triplet: (grasper, grasp, gallbladder) - 711 frames
Graph density: 0.35
Total frames: 1420
Instruments: ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissors']
Targets: ['abdominal_wall_cavity', 'blood_vessel', 'cystic_artery', 'cystic_duct', 'cystic_pedicle', 'fluid', 'gallbladder', 'liver', 'specimen_bag']


In [3]:
# Inspect edges
print('=== All Edges (VID01) ===')
for u, v, k, d in sorted(G_vid01.edges(data=True, keys=True), key=lambda x: -x[3]['frequency']):
    print(f'  {u} --{k}--> {v}  (freq={d["frequency"]}, frames {d["first_frame"]}-{d["last_frame"]})')

=== All Edges (VID01) ===
  grasper --grasp--> gallbladder  (freq=711, frames 0-1433)
  hook --dissect--> gallbladder  (freq=524, frames 493-1430)
  grasper --retract--> liver  (freq=359, frames 1164-1640)
  grasper --grasp--> specimen_bag  (freq=125, frames 1478-1717)
  irrigator --aspirate--> fluid  (freq=117, frames 291-1624)
  grasper --retract--> gallbladder  (freq=113, frames 524-1203)
  bipolar --coagulate--> blood_vessel  (freq=100, frames 325-466)
  clipper --clip--> cystic_duct  (freq=78, frames 698-843)
  grasper --pack--> gallbladder  (freq=27, frames 1512-1540)
  hook --dissect--> cystic_duct  (freq=21, frames 251-271)
  scissors --cut--> cystic_duct  (freq=17, frames 860-876)
  bipolar --coagulate--> cystic_artery  (freq=16, frames 1010-1025)
  irrigator --retract--> liver  (freq=14, frames 1626-1639)
  hook --coagulate--> liver  (freq=9, frames 484-492)
  bipolar --dissect--> cystic_duct  (freq=8, frames 468-475)
  grasper --grasp--> liver  (freq=4, frames 1506-1509)
  i

## Build Graphs for All 3 Videos

In [4]:
print('Building graphs for all 3 videos...')
graphs = build_graphs_for_videos(TRIPLET_DIR, video_ids=['VID01', 'VID05', 'VID40'])

for vid, G in graphs.items():
    print_graph_stats(G)

Building graphs for all 3 videos...
  VID01: 15 nodes, 19 edges
  VID05: 18 nodes, 23 edges
  VID40: 12 nodes, 16 edges

=== VID01 Graph Stats ===
Nodes: 15 (6 instruments + 9 targets)
Edges: 19
Most frequent triplet: (grasper, grasp, gallbladder) - 711 frames
Graph density: 0.35
Total frames: 1420
Instruments: ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissors']
Targets: ['abdominal_wall_cavity', 'blood_vessel', 'cystic_artery', 'cystic_duct', 'cystic_pedicle', 'fluid', 'gallbladder', 'liver', 'specimen_bag']

=== VID05 Graph Stats ===
Nodes: 18 (6 instruments + 12 targets)
Edges: 23
Most frequent triplet: (grasper, retract, gallbladder) - 986 frames
Graph density: 0.32
Total frames: 1736
Instruments: ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissors']
Targets: ['abdominal_wall_cavity', 'cystic_artery', 'cystic_duct', 'cystic_pedicle', 'cystic_plate', 'fluid', 'gallbladder', 'gut', 'liver', 'omentum', 'peritoneum', 'specimen_bag']

=== VID40 Graph Stats ==

## Save Graphs as GEXF

In [5]:
print('Saving graph files...')
for vid, G in graphs.items():
    save_graph(G, GRAPH_DIR / f'{vid}_graph.gexf')

print('\nGraph outputs:', sorted(os.listdir(GRAPH_DIR)))

Saving graph files...
  Saved VID01_graph.gexf
  Saved VID05_graph.gexf
  Saved VID40_graph.gexf

Graph outputs: ['VID01_graph.gexf', 'VID05_graph.gexf', 'VID40_graph.gexf']


## Comparative Summary Table

In [6]:
summary_rows = []
for vid, G in graphs.items():
    s = graph_stats(G)
    mft = s['most_frequent_triplet']
    summary_rows.append({
        'Video': s['video'],
        'Nodes': s['nodes'],
        'Instruments': s['n_instruments'],
        'Targets': s['n_targets'],
        'Edges': s['edges'],
        'Density': s['density'],
        'Frames': s['total_frames'],
        'Top Triplet': f'({mft[0]}, {mft[1]}, {mft[2]})' if mft else '-',
        'Top Freq': mft[3] if mft else 0,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Video,Nodes,Instruments,Targets,Edges,Density,Frames,Top Triplet,Top Freq
0,VID01,15,6,9,19,0.35,1420,"(grasper, grasp, gallbladder)",711
1,VID05,18,6,12,23,0.32,1736,"(grasper, retract, gallbladder)",986
2,VID40,12,5,7,16,0.46,1893,"(grasper, retract, gallbladder)",1479


---

**Next:** Visualize graphs using matplotlib and pyvis.